In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load the data
df = pd.read_csv("gandhinagar_rent_dataset.csv")
df.drop(columns=["Unnamed: 7", "id"], inplace=True)


In [3]:
df.head()

,bhk,property_type,sharing_type,gender_preference,location,area,furnished,ac,fridge,washing_machine,geyser,wifi,water_24h,parking,cooking_allowed,light_bill_included,full_rent
0,3BHK,Flat,Full,Boys,Sargasan,Gandhinagar,Basic,Yes,No,Yes,Yes,No,Yes,Yes,Yes,Yes,41000
1,1BHK,Flat,Full,Family,Sector 3,Gandhinagar,Semi,No,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,9000
2,2BHK,House,Full,Girls,Sector 2,Gandhinagar,Fully,Yes,No,No,No,Yes,Yes,Yes,Yes,No,23000
3,2BHK,Flat,Sharing,Any,Infocity,Gandhinagar,Semi,Yes,No,No,No,Yes,Yes,No,Yes,No,29000
4,2BHK,House,Sharing,Boys,Sector 2,Gandhinagar,Fully,Yes,No,No,Yes,Yes,Yes,No,Yes,Yes,23000


In [4]:
from sklearn.model_selection import StratifiedShuffleSplit
# 2. Stratified Split based on Rent Categories
# Hum rent ko bins mein divide kar rahe hain taaki split balanced ho
df["rent_cat"] = pd.cut(
    df["full_rent"],
    bins=[0, 12000, 18000, 25000, 32000, np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["rent_cat"]):
    strat_train_set = df.loc[train_index].drop("rent_cat", axis=1)
    strat_test_set = df.loc[test_index].drop("rent_cat", axis=1)

# Working on training data
rent_data = strat_train_set.copy()

In [ ]:
# 3. Separate predictors and labels
# Hum 'full_rent' predict kar rahe hain. 
# 'id', 'area', aur 'per_person_rent' ko drop kar rahe hain kyunki ye leakage ya useless features hain.
target = "full_rent"
rent_labels = rent_data[target]
rent_predictors = rent_data.drop([target, "area"], axis=1)

# 4. Separate numerical and categorical columns
num_attribs = rent_predictors.select_dtypes(include=[np.number]).columns.tolist()
cat_attribs = rent_predictors.select_dtypes(exclude=[np.number]).columns.tolist()

In [47]:
cat_attribs

['bhk',
 'property_type',
 'sharing_type',
 'gender_preference',
 'location',
 'furnished',
 'ac',
 'fridge',
 'washing_machine',
 'geyser',
 'wifi',
 'water_24h',
 'parking',
 'cooking_allowed',
 'light_bill_included']

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# 5. Pipelines
# Numerical pipeline (Imputer + Scaler)
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline (OneHotEncoder)
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Full transformation pipeline
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [49]:
# 6. Transform the data
rent_prepared = full_pipeline.fit_transform(rent_predictors)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# 7. Model Training
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(rent_prepared, rent_labels)

# Decision Tree Regressor
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(rent_prepared, rent_labels)

# Random Forest Regressor
forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
forest_reg.fit(rent_prepared, rent_labels)

# 8. Evaluation on Training Data (RMSE)
lin_preds = lin_reg.predict(rent_prepared)
tree_preds = tree_reg.predict(rent_prepared)
forest_preds = forest_reg.predict(rent_prepared)

lin_rmse = root_mean_squared_error(rent_labels, lin_preds)
tree_rmse = root_mean_squared_error(rent_labels, tree_preds)
forest_rmse = root_mean_squared_error(rent_labels, forest_preds)

print("--- Results on Training Data ---")
print(f"Linear Regression RMSE: {lin_rmse:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")  # Isme 0 ke pass aayega (Overfitting)
print(f"Random Forest RMSE: {forest_rmse:.2f}")

# 9. Final Check on Test Set (The Real Test)
X_test = strat_test_set.drop([target,"area"], axis=1)
y_test = strat_test_set[target]
X_test_prepared = full_pipeline.transform(X_test)

final_preds = forest_reg.predict(X_test_prepared)
final_rmse = root_mean_squared_error(y_test, final_preds)
print(f"\nFinal Test Set RMSE (Random Forest): {final_rmse:.2f}")

--- Results on Training Data ---
Linear Regression RMSE: 2694.84
Decision Tree RMSE: 172.12
Random Forest RMSE: 884.95

Final Test Set RMSE (Random Forest): 2281.77
